In [1]:
using QEDprocesses
using QEDcore
using QEDbase

In [7]:
N = 3
FLOAT_T = Float64

OMEGA = FLOAT_T(1000)
MODEL = PerturbativeQED()
PROCESS = ScatteringProcess(
    (Electron(), Photon()),                                                         # incoming particles
    (Electron(), ntuple(_ -> Electron(), N)..., ntuple(_ -> Positron(), N)...),     # outgoing particles
    (SyncedSpin(1), AllPol()),
    (SyncedSpin(1), ntuple(_ -> SyncedSpin(1), N)..., ntuple(_ -> SyncedSpin(2), N)...),
)
IN_PSL = TwoBodyTargetSystem()
PSL = FlatPhaseSpaceLayout(IN_PSL)

FlatPhaseSpaceLayout{TwoBodyTargetSystem{Energy{2}}}(TwoBodyTargetSystem{Energy{2}}(Energy{2}()))

In [8]:
psp = PhaseSpacePoint(
    PROCESS,
    MODEL,
    PSL,
    (OMEGA,),
    tuple((rand(FLOAT_T) for _ in 1:phase_space_dimension(PROCESS, MODEL, PSL))...)
)

PhaseSpacePoint:
    process: generic QED process "ek -> eeeeppp"
    model: perturbative QED
    phase space layout: FlatPhaseSpaceLayout{TwoBodyTargetSystem{Energy{2}}}(TwoBodyTargetSystem{Energy{2}}(Energy{2}()))
    incoming particles:
     -> incoming electron: [1.0, 0.0, 0.0, 0.0]
     -> incoming photon: [1000.0, 0.0, 0.0, 1000.0]
    outgoing particles:
     -> outgoing electron: [207.6224026462538, 0.43757192878588774, 6.894857123685875, 207.50501573854598]
     -> outgoing electron: [7.354426535591665, -1.5483035241308003, -0.6180850864333667, 7.092835588853554]
     -> outgoing electron: [27.466763587457525, 1.4321449732253335, -1.9534475613594946, 27.34147226046846]
     -> outgoing electron: [226.22430176105385, -0.14879871532786101, 2.7125249967977902, 226.20577971033458]
     -> outgoing positron: [134.60469987431404, -4.799196463030999, -8.95185326572109, 134.2172018210497]
     -> outgoing positron: [98.61709642753141, -1.0376788334001652, -1.2147958615103585, 98.59908

In [ ]:
QEDbase.unsafe_differential_cross_section(psp)

┌ Info: built graph
└ @ QEDprocesses /home/antonr/repos/QEDprocesses.jl/src/processes/generic_process/perturbative/cross_section.jl:30
┌ Info: Graph:
│   Nodes: Total: 53054, QEDFeynmanDiagrams.ComputeTask_BaseState: 18, ComputableDAGs.DataTask: 26673, 
│          QEDFeynmanDiagrams.ComputeTask_CollectTriples: 8, QEDFeynmanDiagrams.ComputeTask_TripleNegated: 13056, QEDFeynmanDiagrams.ComputeTask_CollectPairs: 1448, 
│          QEDFeynmanDiagrams.ComputeTask_PropagatePairs: 1448, QEDFeynmanDiagrams.ComputeTask_SpinPolCumulation: 1, QEDFeynmanDiagrams.ComputeTask_Pair: 3530, 
│          QEDFeynmanDiagrams.ComputeTask_PairNegated: 454, QEDFeynmanDiagrams.ComputeTask_Triple: 6144, QEDFeynmanDiagrams.ComputeTask_Propagator: 274
│   Edges: 118329
│   Total Compute Effort: 0.0
│   Total Data Transfer: 0.0
│   Total Compute Intensity: 0.0
└ @ QEDprocesses /home/antonr/repos/QEDprocesses.jl/src/processes/generic_process/perturbative/cross_section.jl:31


In [ ]:
using ProgressMeter

N_EVENTS = 1_000_000
EVENTS = Matrix{FLOAT_T}(undef, (N_EVENTS, (phase_space_dimension(PROCESS, MODEL, PSL) + 1)))

@showprogress for i in 1:N_EVENTS
    coords = tuple((rand(FLOAT_T) for _ in 1:phase_space_dimension(PROCESS, MODEL, PSL))...)
    psp = PhaseSpacePoint(
        PROCESS,
        MODEL,
        PSL,
        (OMEGA,),
        coords,
    )

    diff_cs = unsafe_differential_cross_section(psp)
    EVENTS[i, :] = [diff_cs, coords...]
end

In [ ]:
using JLD2

@save "$(N)_pair_shower_events.jld2" EVENTS